# 台灣華語 Whisper small＋LoRA：Google Colab GPU

先以免費 GPU 配額嘗試；Google 不保證提供 GPU、特定型號或連線時長。這個 Notebook 不建立付費雲端機器、不發布模型。雲端 GPU 執行訓練，Cloudflare 保留網站／房間服務。

目前資料仍在下載／核對階段，**不代表已有可訓練資料**。每個階段分開執行，`RUN_TRAINING` 預設 False。訓練入口會重新做資料與 GPU 檢查，未通過不訓練。請先從「執行階段 → 變更執行階段類型」選擇 GPU。


In [ ]:
from pathlib import Path
import subprocess, sys
REPO = Path('/content/Meeting-Live-Captions')
REPO_REF = 'main'  # 重跑既有版本時改為該 run-lock.json 記錄的 code_commit
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/timo893131/Meeting-Live-Captions.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'checkout',REPO_REF],check=True)
print('程式版本：',subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())
subprocess.run([sys.executable,'-m','pip','install','-r',str(REPO/'training/requirements-colab.txt')],check=True)


In [ ]:
import torch
print('PyTorch:',torch.__version__,'CUDA:',torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError('尚無 CUDA GPU，請切換 Colab GPU 執行階段；未開始訓練。')
print('GPU:',torch.cuda.get_device_name(0))
print('GPU 記憶體 GiB:',round(torch.cuda.get_device_properties(0).total_memory/2**30,2))


## 私人資料與輸出位置

掛載 Drive 由你在 Colab 完成。上傳可訓練的 WAV 與三份人工審核 manifest，音檔路徑相對 DATA_ROOT。保持原音，使用 16 kHz、mono、PCM16、5–30 秒工作副本。

現有爬蟲的 pending／unassigned manifest **不能直接用於訓練**。格式與必填審核欄位見 repo 的 docs/cloud-training.md。資料權利是 training（train）或 internal_evaluation（validation/test），不能只用「公開可讀」取代。未知講員預設擋下；若研究設定允許，須明確修改設定並揭露隔離限制。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import json
from datetime import datetime, timezone
DATA_ROOT = Path('/content/drive/MyDrive/meeting-live-captions/data')
RUN_ID = datetime.now(timezone.utc).strftime('whisper-small-smoke-%Y%m%dT%H%M%SZ')
RUN_DIR = Path('/content/drive/MyDrive/meeting-live-captions/runs')/RUN_ID
CONFIG = json.loads((REPO/'training/cloud-whisper-lora.json').read_text())
CONFIG['data']['root'] = str(DATA_ROOT)
CONFIG['training']['mode'] = 'smoke'  # 20 步，只驗證流程；pilot 也需獨立版本與實測評估
CONFIG_PATH = Path('/content/cloud-training-config.json')
CONFIG_PATH.write_text(json.dumps(CONFIG,ensure_ascii=False,indent=2))
print('資料：',DATA_ROOT,'輸出：',RUN_DIR)


## 可選：雲端資料蒐集

Colab 網路與 Codex 雲端環境分開；這裡仍可能遇到來源限制，不能保證可下載。以下只跑原有有上限的 15 段資料流程，並不產生 train-ready 標籤。範例預設不執行，避免與已審核的 DATA_ROOT 混用。


In [ ]:
RUN_COLLECTION = False
if RUN_COLLECTION:
    subprocess.run([sys.executable,str(REPO/'scripts/tw_speech_pilot.py'),'--root','/content/taiwan-pilot','--prepare'],cwd=REPO,check=True)
else:
    print('未執行資料下載；準備好授權與審核後再執行。')


## 訓練前資料檢查

同一 recording、speaker、原音／PCM hash 或完全相同文字不可跨集合。近似去重需有實際審核結果；不從欄位名稱推定品質。test 只作資料洩漏檢查，不傳進 Trainer 或 early stopping。


In [ ]:
subprocess.run([sys.executable,str(REPO/'scripts/cloud_training_data.py'),'--config',str(CONFIG_PATH)],cwd=REPO,check=True)


## 小規模試訓練與中斷續跑

預設 LoRA r=8、batch=1、gradient accumulation=4、20 steps、每10步評估及存checkpoint。30分鐘為每次 train 呼叫的逐步檢查上限，不包含前置下載／資料檢查／基線與後測評估。先驗證流程，再決定正式步數；20步不保證品質提升。

`RUN_TRAINING=True` 才會呼叫訓練。首次解析模型 commit 後寫入 run-lock.json，resume 必須使用相同程式、資料hash、設定及主要套件版本。若中斷，將 RUN_DIR 指向原目錄並設定 RESUME=True；沒有 checkpoint 時需使用新 run。成功完成的 run 不覆寫。


In [ ]:
RUN_TRAINING = False
RESUME = False
if RUN_TRAINING:
    command=[sys.executable,str(REPO/'scripts/train_cloud_whisper.py'),'--config',str(CONFIG_PATH),'--run-dir',str(RUN_DIR)]
    if RESUME:
        command.append('--resume')
    subprocess.run(command,cwd=REPO,check=True)
else:
    print('尚未開始訓練。請先完成資料檢查，再明確啟用 RUN_TRAINING。')


## 讀取結果

checkpoint、adapter、processor 與驗證結果保存於 Drive。before/after CER 採同一個雲端正規化版本；它是 validation 指標，不能當作未見test或真實即時延遲。最終還需選定候選後獨立跑 test、正常速度音訊／麥克風測試，並檢查轉回 CTranslate2/int8 的結果。

目前 Notebook 已做靜態與資料 gate 測試，**尚未在 Colab GPU 實跑**；不宣稱已有 checkpoint 或訓練改善。當前來源沒有正式 train-ready 資料。


In [ ]:
report=RUN_DIR/'final-report.json'
if report.exists():
    print(report.read_text())
else:
    print('尚無完成報告。若已試跑，可查看 RUN_DIR/checkpoints 與 attempts。')
